In [1]:
import zipfile
import os
import datetime
import pandas as pd

DATA_DIR = "data"
DIVVY_DIR = f"{DATA_DIR}/divvy"

In [2]:
def extractFromZip(dir, file):
    with zipfile.ZipFile(f"{dir}/{file}", mode="r") as zf:
        with zf.open(file.split(".zip")[0]) as f:
            return pd.read_csv(f)

def buildDivvyDataset(dir):
    files = os.listdir(dir)
    df = pd.DataFrame()
    for f in files:
        df = pd.concat([df, extractFromZip(dir, f)])
    return df

In [10]:
divvy_df = buildDivvyDataset(DIVVY_DIR)
weather_df = pd.concat([pd.read_csv(DATA_DIR + "/2025-chicago-weather.csv", skiprows=2), pd.read_csv(DATA_DIR + "/2026-chicago-weather.csv", skiprows=2)])

In [11]:
weather_df["time"] = pd.to_datetime(weather_df["time"])
weather_df["date"] = weather_df["time"].dt.date

daily_weather = (
    weather_df.groupby("date")
    .agg(
        avg_temp=("temperature_2m (°C)", "mean"),
        max_temp=("temperature_2m (°C)", "max"),
        min_temp=("temperature_2m (°C)", "min"),
        cumulative_rain=("rain (mm)", "sum"),
        cumulative_snow=("snowfall (cm)", "sum"),
    )
    .reset_index()
)

In [22]:
divvy_df['started_at'] = pd.to_datetime(divvy_df['started_at'])
divvy_df['ended_at'] = pd.to_datetime(divvy_df['ended_at'])
divvy_df = divvy_df[divvy_df['started_at'] <= divvy_df['ended_at']].reset_index(drop=True)
divvy_counts = divvy_df.groupby(divvy_df['started_at'].dt.date).size()

In [23]:
divvy_df

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,7569BC890583FCD7,classic_bike,2025-01-21 17:23:54.538,2025-01-21 17:37:52.015,Wacker Dr & Washington St,KA1503000072,McClurg Ct & Ohio St,TA1306000029,41.883143,-87.637242,41.892592,-87.617289,member
1,013609308856B7FC,electric_bike,2025-01-11 15:44:06.795,2025-01-11 15:49:11.139,Halsted St & Wrightwood Ave,TA1309000061,Racine Ave & Belmont Ave,TA1308000019,41.929147,-87.649153,41.939743,-87.658865,member
2,EACACD3CE0607C0D,classic_bike,2025-01-02 15:16:27.730,2025-01-02 15:28:03.230,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member
3,EAA2485BA64710D3,classic_bike,2025-01-23 08:49:05.814,2025-01-23 08:52:40.047,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member
4,7F8BE2471C7F746B,electric_bike,2025-01-16 08:38:32.338,2025-01-16 08:41:06.767,Southport Ave & Waveland Ave,13235,Southport Ave & Roscoe St,13071,41.948226,-87.664071,41.943739,-87.664020,member
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6209234,1ADE06C58744ECAB,electric_bike,2026-03-02 09:11:49.421,2026-03-02 09:16:37.134,Western Ave & Lunt Ave,CHI00801,Clark St & Lunt Ave,CHI00642,42.008594,-87.690492,42.009011,-87.674112,member
6209235,87819D83AF437454,classic_bike,2026-03-05 07:38:53.806,2026-03-05 07:44:13.659,Western Ave & Lunt Ave,CHI00801,Clark St & Lunt Ave,CHI00642,42.008594,-87.690492,42.009011,-87.674112,member
6209236,7E910EFCCBCF8440,electric_bike,2026-03-25 18:27:59.727,2026-03-25 18:35:06.921,DuSable Lake Shore Dr & Belmont Ave,CHI00480,Southport Ave & Waveland Ave,CHI00336,41.940775,-87.639192,41.948226,-87.664071,casual
6209237,577CC5F21A42A372,electric_bike,2026-03-09 16:26:54.043,2026-03-09 16:29:55.413,Western Ave & Lunt Ave,CHI00801,Clark St & Lunt Ave,CHI00642,42.008594,-87.690492,42.009011,-87.674112,member


In [25]:
ridership_df = divvy_counts.reset_index(name="ride_count")
ridership_df = ridership_df[
    ridership_df["started_at"] >= datetime.date(2025, 1, 1)
]
ridership_df["started_at"] = pd.to_datetime(ridership_df["started_at"]).dt.date

df = pd.merge(
    ridership_df,
    daily_weather,
    left_on="started_at",
    right_on="date",
    how="left",
).drop(columns=["date"])

In [26]:
df

,started_at,ride_count,avg_temp,max_temp,min_temp,cumulative_rain,cumulative_snow
0,2025-01-01,3562,-1.329167,0.8,-3.1,0.0,0.07
1,2025-01-02,4749,-2.433333,-0.3,-4.2,0.0,0.14
2,2025-01-03,4200,-5.116667,-3.4,-7.4,0.0,0.00
3,2025-01-04,3198,-7.991667,-5.1,-11.0,0.0,0.00
4,2025-01-05,2933,-6.391667,-2.3,-9.0,0.1,0.35
...,...,...,...,...,...,...,...
450,2026-03-27,9076,1.420833,4.4,-0.4,0.0,0.07
451,2026-03-28,11336,1.945833,8.0,-1.9,0.0,0.00
452,2026-03-29,13611,8.975000,16.3,4.3,0.0,0.00
453,2026-03-30,21316,15.991667,27.1,10.0,0.0,0.00


In [28]:
df.to_csv('data/daily_ridership_with_weather.csv')